# 05 — Federated Learning (Simulated FedAvg): Taxonomy Classification\n
\n
Simulate federated clients by partitioning the FishNet training set into shards (stratified by species).\n
\n
Train via FedAvg and compare to centralized training.\n
\n
Implementation: minimal pure-PyTorch FedAvg loop (device-flexible).\n
Source reference (general FL concepts): https://flower.ai/docs/

In [1]:
from pathlib import Path
import sys

import numpy as np
import torch

ROOT = Path('..').resolve()
sys.path.append(str((ROOT / 'src').resolve()))

MANIFEST_DIR = (ROOT / 'data' / 'manifests').resolve()
OUT_DIR = (ROOT / 'outputs' / 'federated_fedavg').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device(
    'cuda'
    if torch.cuda.is_available()
    else 'mps'
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
    else 'cpu'
)
print('device:', device)
print('OUT_DIR:', OUT_DIR)


device: cuda
OUT_DIR: D:\Fish Codes\outputs\federated_fedavg


In [2]:
from torch.utils.data import DataLoader, Subset

from fish_ai.data.jsonl import read_jsonl
from fish_ai.data.taxonomy_dataset import FishTaxonomyDataset
from fish_ai.models.taxonomy_classifier import TaxonomyClassifier, TaxonomyHeadSizes
from fish_ai.train.fedavg import (
    FedAvgConfig,
    average_state_dicts,
    select_clients,
    split_indices_stratified,
)
from fish_ai.train.taxonomy_train import TrainConfig, build_label_maps, evaluate, train_one_epoch
from fish_ai.utils.run_logging import write_csv, write_json

train_manifest = MANIFEST_DIR / 'fishnet_taxonomy_train.jsonl'
val_manifest = MANIFEST_DIR / 'fishnet_taxonomy_val.jsonl'
print(train_manifest.exists(), val_manifest.exists())


True True


In [3]:
# Build label maps from FULL training set
train_rows = []
for r in read_jsonl(train_manifest):
    tax = r['taxonomy']
    train_rows.append({'family': tax['family'], 'genus': tax['genus'], 'species': tax['species']})

maps = build_label_maps(train_rows)

sizes = TaxonomyHeadSizes(
    n_family=len(maps['family']),
    n_genus=len(maps['genus']),
    n_species=len(maps['species']),
)

sizes


TaxonomyHeadSizes(n_family=91, n_genus=199, n_species=100)

In [4]:
# Datasets
ds_train_full = FishTaxonomyDataset(train_manifest, image_size=224, augment=True, max_side_before_square=512)
ds_val = FishTaxonomyDataset(val_manifest, image_size=224, augment=False, max_side_before_square=512)

# Client partitions (stratified by species)
species_ids = [maps['species'][r['species']] for r in train_rows]

fed_cfg = FedAvgConfig(
    num_clients=10,
    num_rounds=10,
    clients_per_round=None,
    local_epochs=1,
    seed=42,
)
client_splits = split_indices_stratified(species_ids, num_clients=fed_cfg.num_clients, seed=fed_cfg.seed)

[(i, len(s)) for i, s in enumerate(client_splits)][:5], sum(len(s) for s in client_splits)


([(0, 3814), (1, 3794), (2, 3783), (3, 3774), (4, 3764)], 37700)

In [5]:
# Global model (start from transfer learning weights)
global_model = TaxonomyClassifier(sizes, backbone='efficientnet_b0', pretrained=True).to(device)
cfg = TrainConfig(num_epochs=fed_cfg.local_epochs, batch_size=64, num_workers=2, lr=3e-4)

dl_val = DataLoader(ds_val, batch_size=64, shuffle=False, num_workers=2)

history = []
for rnd in range(1, fed_cfg.num_rounds + 1):
    selected = select_clients(
        fed_cfg.num_clients,
        fed_cfg.clients_per_round,
        round_idx=rnd,
        seed=fed_cfg.seed,
    )
    client_states = []
    client_weights = []
    client_train_losses = []

    for cid in selected:
        idxs = client_splits[cid]
        if len(idxs) == 0:
            continue

        client_ds = Subset(ds_train_full, idxs)
        client_dl = DataLoader(
            client_ds,
            batch_size=cfg.batch_size,
            shuffle=True,
            num_workers=cfg.num_workers,
        )

        client_model = TaxonomyClassifier(sizes, backbone='efficientnet_b0', pretrained=False).to(device)
        client_model.load_state_dict(global_model.state_dict())
        opt = torch.optim.AdamW(client_model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

        tr_sum = None
        for _ in range(fed_cfg.local_epochs):
            tr = train_one_epoch(client_model, client_dl, opt, device=device, maps=maps)
            if tr_sum is None:
                tr_sum = {key: 0.0 for key in tr}
            for key in tr_sum:
                tr_sum[key] += tr[key]
        ne = max(fed_cfg.local_epochs, 1)
        tr_avg = {key: tr_sum[key] / ne for key in tr_sum}

        client_states.append({k: v.detach().cpu() for k, v in client_model.state_dict().items()})
        client_weights.append(float(len(idxs)))
        client_train_losses.append(tr_avg)

    wsum = sum(client_weights)
    train_weighted = {}
    for k in ['loss_total', 'loss_family', 'loss_genus', 'loss_species']:
        train_weighted[f'train_{k}_weighted_mean'] = (
            sum(client_train_losses[i][k] * client_weights[i] for i in range(len(client_weights))) / wsum
            if wsum > 0
            else 0.0
        )
    for k in ['train_species_acc_top1', 'train_genus_acc_top1', 'train_family_acc_top1']:
        train_weighted[f'{k}_weighted_mean'] = (
            sum(client_train_losses[i][k] * client_weights[i] for i in range(len(client_weights))) / wsum
            if wsum > 0
            else 0.0
        )

    avg_state = average_state_dicts(client_states, weights=client_weights)
    global_model.load_state_dict(avg_state, strict=True)
    global_model.to(device)

    val_metrics = evaluate(global_model, dl_val, device=device, maps=maps)
    row = {'round': rnd, 'num_selected_clients': len(selected), **train_weighted}
    row['val_loss_total'] = val_metrics['loss_total']
    for lvl in ['family', 'genus', 'species']:
        for k, v in val_metrics[lvl].items():
            row[f'val_{lvl}_{k}'] = v
    history.append(row)
    print(row)

ckpt_path = OUT_DIR / 'fedavg_global_model.pt'
torch.save(
    {
        'model_state': global_model.state_dict(),
        'backbone': global_model.backbone_name,
        'maps': maps,
        'fed_cfg': fed_cfg.__dict__,
    },
    ckpt_path,
)

write_csv(OUT_DIR / 'history.csv', history)
write_json(OUT_DIR / 'val_metrics_last.json', val_metrics)
print('saved:', ckpt_path)
print('wrote:', OUT_DIR / 'history.csv')


{'round': 1, 'num_selected_clients': 10, 'val_species_acc_top1': 0.9682425978987583, 'val_species_f1_macro': 0.012955023226391827, 'val_genus_acc_top1': 0.1361031518624642, 'val_family_acc_top1': 0.25620821394460364}
{'round': 2, 'num_selected_clients': 10, 'val_species_acc_top1': 0.9684813753581661, 'val_species_f1_macro': 0.017781771998639468, 'val_genus_acc_top1': 0.3526743075453677, 'val_family_acc_top1': 0.45367717287488063}
{'round': 3, 'num_selected_clients': 10, 'val_species_acc_top1': 0.9689589302769819, 'val_species_f1_macro': 0.03187645768539202, 'val_genus_acc_top1': 0.4777936962750716, 'val_family_acc_top1': 0.565425023877746}
{'round': 4, 'num_selected_clients': 10, 'val_species_acc_top1': 0.970152817574021, 'val_species_f1_macro': 0.04704364155717484, 'val_genus_acc_top1': 0.5470391595033429, 'val_family_acc_top1': 0.623447946513849}
{'round': 5, 'num_selected_clients': 10, 'val_species_acc_top1': 0.9713467048710601, 'val_species_f1_macro': 0.0696565934065934, 'val_genus